# Grupo 12 — Pré-processamento WLASL-100 (Gestos Dinâmicos)

Este notebook processa os dados **já pré-processados do WLASL-100 do Kaggle**, que vêm em duas formas:
- `preprocessing/pose/<split>/<word>/<video_id>/*.png` — imagens com landmarks desenhados (pose)
- `preprocessing/frame/<split>/<word>/<video_id>/*.png` — frames reais do vídeo

**O que fazemos aqui:**
Vamos re-extrair os landmarks com MediaPipe Holistic a partir dos **frames reais** (`frame/`) e guardar em arrays NumPy — um array por vídeo, com shape `(T, N_FEATURES)` — prontos para treinar a LSTM no notebook seguinte.

**Porquê não usar as imagens de pose do dataset?**  
As imagens PNG de pose já têm os landmarks *desenhados* como pixeis — não são coordenadas numéricas. Precisamos das coordenadas para alimentar a LSTM, por isso re-extraímos do frame real.

---
### Estrutura esperada do dataset:
```
wlasl100/
  preprocessing/
    frame/
      train/  test/  val/
        <word>/
          <video_id>/
            frame_000.png  frame_001.png  ...
    pose/     (não usado — apenas frames reais)
```

In [8]:
# ── Dependências ──────────────────────────────────────────────────────────
# No Google Colab: já estão instaladas. Localmente, descomenta se necessário.
# !pip install mediapipe opencv-python numpy tqdm --quiet
import subprocess, sys
for pkg in ['mediapipe', 'opencv-python', 'numpy', 'tqdm']:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q',
                           '--break-system-packages'],
                          stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
print('✅ Dependências OK')

✅ Dependências OK


In [9]:
import os
import cv2
import numpy as np
import mediapipe as mp
from tqdm import tqdm

print('MediaPipe versão:', mp.__version__)

MediaPipe versão: 0.10.21


In [10]:
# ════════════════════════════════════════════════════════════════════
# CONFIGURAÇÃO — ajusta WLASL_ROOT ao caminho do teu dataset
# ════════════════════════════════════════════════════════════════════

# ---- LOCAL ----
# WLASL_ROOT = '/caminho/para/wlasl100'          # pasta que contém 'preprocessing/'

# ---- GOOGLE COLAB (após fazer upload ou mount do Drive) ----
# from google.colab import drive
# drive.mount('/content/drive')
# WLASL_ROOT = '/content/drive/MyDrive/wlasl100'

WLASL_ROOT = 'wlasl100'                          # ← muda aqui

FRAME_DIR  = os.path.join(WLASL_ROOT, 'preprocessing', 'frame')
OUTPUT_DIR = 'outputs_dynamic'
os.makedirs(OUTPUT_DIR, exist_ok=True)

SPLITS = ['train', 'test', 'val']

# Número de frames por vídeo (padding/truncamento para LSTM de comprimento fixo)
SEQ_LEN = 30          # 30 frames ~= 1 s a 30 fps — padrão na literatura

# MediaPipe Holistic extrai: 33 pose + 21 mão esq + 21 mão dir + 468 face
# Usamos apenas pose (33×4=132) + ambas as mãos (21×3×2=126) = 258 features por frame
# A face é descartada para reduzir dimensionalidade (não crucial para sinais)
N_POSE    = 33 * 4    # x, y, z, visibility
N_HAND    = 21 * 3    # x, y, z por mão
N_FEATURES = N_POSE + 2 * N_HAND  # 132 + 126 = 258

print(f'Frame dir: {FRAME_DIR}')
print(f'Output dir: {OUTPUT_DIR}')
print(f'SEQ_LEN = {SEQ_LEN}, N_FEATURES = {N_FEATURES}')

Frame dir: wlasl100/preprocessing/frame
Output dir: outputs_dynamic
SEQ_LEN = 30, N_FEATURES = 258


In [11]:
# ════════════════════════════════════════════════════════════════════
# Verificação da estrutura do dataset
# ════════════════════════════════════════════════════════════════════

if not os.path.isdir(FRAME_DIR):
    raise FileNotFoundError(
        f"Pasta não encontrada: {FRAME_DIR}\n"
        "Certifica-te de que WLASL_ROOT aponta para a pasta correta."
    )

for split in SPLITS:
    split_dir = os.path.join(FRAME_DIR, split)
    if os.path.isdir(split_dir):
        words = [d for d in os.listdir(split_dir) if os.path.isdir(os.path.join(split_dir, d))]
        print(f'  {split:5s}: {len(words):3d} palavras')
    else:
        print(f'  {split}: ⚠️  pasta não encontrada')

  train: 100 palavras
  test : 100 palavras
  val  : 100 palavras


In [12]:
# ════════════════════════════════════════════════════════════════════
# Funções de extração de landmarks
# ════════════════════════════════════════════════════════════════════

def extract_holistic_features(frame_bgr, holistic):
    """
    Dada uma frame BGR e uma instância do MediaPipe Holistic,
    devolve um vetor de N_FEATURES valores.

    Se um componente não for detetado (ex.: mão esquerda fora de campo),
    preenche com zeros — estratégia de interpolação baseline.
    """
    frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    frame_rgb.flags.writeable = False
    results = holistic.process(frame_rgb)
    frame_rgb.flags.writeable = True

    # --- Pose (33 landmarks × 4: x, y, z, visibility) ---
    if results.pose_landmarks:
        pose = np.array(
            [[lm.x, lm.y, lm.z, lm.visibility] for lm in results.pose_landmarks.landmark],
            dtype=np.float32
        ).flatten()  # (132,)
    else:
        pose = np.zeros(N_POSE, dtype=np.float32)

    # --- Mão esquerda (21 × 3) ---
    if results.left_hand_landmarks:
        lh = np.array(
            [[lm.x, lm.y, lm.z] for lm in results.left_hand_landmarks.landmark],
            dtype=np.float32
        ).flatten()  # (63,)
    else:
        lh = np.zeros(N_HAND, dtype=np.float32)

    # --- Mão direita (21 × 3) ---
    if results.right_hand_landmarks:
        rh = np.array(
            [[lm.x, lm.y, lm.z] for lm in results.right_hand_landmarks.landmark],
            dtype=np.float32
        ).flatten()  # (63,)
    else:
        rh = np.zeros(N_HAND, dtype=np.float32)

    return np.concatenate([pose, lh, rh])  # (258,)


def frames_to_sequence(frame_paths, holistic, seq_len=SEQ_LEN):
    """
    Lê uma lista de frames PNG, extrai landmarks de cada uma e
    devolve um array (seq_len, N_FEATURES) com padding/truncamento.

    Estratégia de padding:
      - Se o vídeo tem mais frames que seq_len: pega as do meio (center-crop temporal)
      - Se tem menos: repete a última frame (padding por repetição)
    """
    vectors = []
    for fp in sorted(frame_paths):   # garante ordem correta
        img = cv2.imread(fp)
        if img is None:
            continue
        vec = extract_holistic_features(img, holistic)
        vectors.append(vec)

    if len(vectors) == 0:
        return np.zeros((seq_len, N_FEATURES), dtype=np.float32)

    arr = np.stack(vectors, axis=0)  # (T, N_FEATURES)
    T   = arr.shape[0]

    if T >= seq_len:
        # Center-crop temporal: pega as frames do meio
        start = (T - seq_len) // 2
        return arr[start : start + seq_len]
    else:
        # Padding por repetição da última frame
        pad = np.tile(arr[-1:], (seq_len - T, 1))
        return np.vstack([arr, pad]).astype(np.float32)


print('✅ Funções de extração definidas')

✅ Funções de extração definidas


In [13]:
# ════════════════════════════════════════════════════════════════════
# Processamento principal — um split de cada vez
#
# Saídas por split:
#   outputs_dynamic/X_train.npy  shape: (N_videos, SEQ_LEN, N_FEATURES)
#   outputs_dynamic/y_train.npy  shape: (N_videos,)  — labels string
# ════════════════════════════════════════════════════════════════════

# Instanciar Holistic UMA vez (não dentro do loop — é pesado)
mp_holistic = mp.solutions.holistic
holistic = mp_holistic.Holistic(
    static_image_mode=True,          # True para frames isoladas
    model_complexity=1,
    enable_segmentation=False,
    refine_face_landmarks=False,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
)

stats = {}   # para diagnóstico

for split in SPLITS:
    split_dir = os.path.join(FRAME_DIR, split)
    if not os.path.isdir(split_dir):
        print(f'⚠️  {split} não existe, a ignorar.')
        continue

    words = sorted([d for d in os.listdir(split_dir)
                    if os.path.isdir(os.path.join(split_dir, d))])

    X_list, y_list = [], []
    n_empty = 0

    for word in tqdm(words, desc=f'  {split}', unit='palavra'):
        word_dir = os.path.join(split_dir, word)
        video_ids = [v for v in os.listdir(word_dir)
                     if os.path.isdir(os.path.join(word_dir, v))]

        for vid in video_ids:
            vid_dir = os.path.join(word_dir, vid)
            frame_paths = [
                os.path.join(vid_dir, f)
                for f in os.listdir(vid_dir)
                if f.lower().endswith('.jpg')
            ]

            if not frame_paths:
                n_empty += 1
                continue

            seq = frames_to_sequence(frame_paths, holistic, SEQ_LEN)
            X_list.append(seq)
            y_list.append(word)

    if not X_list:
        print(f'  ⚠️  {split}: nenhuma sequência extraída!')
        continue

    X = np.stack(X_list, axis=0)  # (N, SEQ_LEN, N_FEATURES)
    y = np.array(y_list)          # (N,)

    np.save(os.path.join(OUTPUT_DIR, f'X_{split}.npy'), X)
    np.save(os.path.join(OUTPUT_DIR, f'y_{split}.npy'), y)

    stats[split] = {'n_videos': len(X_list), 'n_empty': n_empty,
                    'shape': X.shape, 'n_classes': len(set(y_list))}
    print(f'  ✅ {split}: {len(X_list)} sequências, {len(set(y_list))} classes '
          f'| vazios ignorados: {n_empty}')

holistic.close()

print('\n=== Resumo ===')
for split, s in stats.items():
    print(f'  {split:5s}  videos={s["n_videos"]:4d}  classes={s["n_classes"]:3d}  shape={s["shape"]}')

I0000 00:00:1774795118.479134    9719 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1774795118.480110   10971 gl_context.cc:369] GL version: 3.2 (OpenGL ES 3.2 Mesa 23.2.1-1ubuntu3.1~22.04.3), renderer: Mesa Intel(R) Graphics (ADL GT2)
  train:   0%|          | 0/100 [00:00<?, ?palavra/s]W0000 00:00:1774795118.520300   10954 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1774795118.544863   10967 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1774795118.546894   10965 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1774795118.547157   10962 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signatu

  ✅ train: 1440 sequências, 100 classes | vazios ignorados: 0


  test: 100%|██████████| 100/100 [03:05<00:00,  1.85s/palavra]


  ✅ test: 258 sequências, 100 classes | vazios ignorados: 0


  val: 100%|██████████| 100/100 [04:04<00:00,  2.44s/palavra]

  ✅ val: 337 sequências, 100 classes | vazios ignorados: 0

=== Resumo ===
  train  videos=1440  classes=100  shape=(1440, 30, 258)
  test   videos= 258  classes=100  shape=(258, 30, 258)
  val    videos= 337  classes=100  shape=(337, 30, 258)


In [14]:
# ════════════════════════════════════════════════════════════════════
# Diagnóstico rápido — distribuição de classes no treino
# ════════════════════════════════════════════════════════════════════
import collections

y_train = np.load(os.path.join(OUTPUT_DIR, 'y_train.npy'), allow_pickle=True)
counter = collections.Counter(y_train)

print(f'Classes no treino: {len(counter)}')
print(f'Média de vídeos/classe: {np.mean(list(counter.values())):.1f}')
print(f'Min: {min(counter.values())}  Max: {max(counter.values())}')
print()
print('As 10 classes com menos exemplos:')
for word, cnt in sorted(counter.items(), key=lambda x: x[1])[:10]:
    print(f'  {word:20s}: {cnt}')

print()
print('✅ Pré-processamento concluído!')
print(f'   Ficheiros guardados em: {OUTPUT_DIR}/')

Classes no treino: 100
Média de vídeos/classe: 14.4
Min: 12  Max: 30

As 10 classes com menos exemplos:
  africa              : 12
  birthday            : 12
  but                 : 12
  cheat               : 12
  corn                : 12
  full                : 12
  how                 : 12
  language            : 12
  letter              : 12
  right               : 12

✅ Pré-processamento concluído!
   Ficheiros guardados em: outputs_dynamic/
